# Automate Channel Counting (replaces the manual QGIS "Line Intersections" step)

**Inputs**
- `01_Concentric_Cercle.geojson` — 41 ring boundaries at 5 km intervals (distance = 5000, 10000, ..., ~205000 m), each ring is already a circle *boundary line* (not a filled disk).
- `CL_FID08_YYYY.geojson` — one file per year (1988–2021), each containing that year's channel centerline network as line geometries.

**What this notebook does**
For every year, it merges all channel-line segments into a single geometry, then intersects each ring boundary with that geometry. The **number of intersection points on a ring = AF(d)** at that ring's distance — this is exactly what QGIS's "Line Intersections" tool gives you, just automated across every ring and every year in one pass.

**Output**
One Excel file, one sheet per year, columns `distance` and `AF` — ready to feed straight into the normalization / entropy / PSD pipeline (no more need to build raw repeated-distance lists first).

In [1]:
import geopandas as gpd      # reading geojson, handling CRS and geometry operations
import pandas as pd          # building result tables, writing Excel
import re                    # extracting the year number from each filename
from pathlib import Path     # convenient file-path handling and folder scanning

---
## Step 1 — Set paths

In [8]:
CIRCLE_PATH = r"G:\Cahnnel Complexity Ganges\CL_Final\01_Concentric_Cercle.geojson"   # ring boundaries
CENTERLINE_DIR = Path(r"G:\Cahnnel Complexity Ganges\CL_Final")                        # folder with yearly files
OUTPUT_XLSX = r"G:\Cahnnel Complexity Ganges\AF_Counts_AllYears.xlsx"                   # where results are saved

---
## Step 2 — Load the concentric circle rings and check the CRS

The CRS (coordinate reference system) MUST be a projected system in meters (e.g. a UTM zone) — otherwise distances won't be in real km and the intersection geometry will be wrong.

In [9]:
circles = gpd.read_file(CIRCLE_PATH)     # load the 41 ring-boundary line features
print("CRS of circle file:", circles.crs)
print("Number of rings:", len(circles))
print(circles[["ringId", "distance", "len"]].head())

CRS of circle file: EPSG:32645
Number of rings: 41
   ringId  distance         len
0       1    5000.0   31395.267
1       2   10000.0   62790.505
2       3   15000.0   94185.685
3       4   20000.0  125580.778
4       5   25000.0  156975.755


---
## Step 3 — Discover all yearly centerline files automatically

Scans the folder for files named `CL_FID08_YYYY.geojson` and extracts the year from the filename, so you don't have to list all 34 years by hand.

In [10]:
pattern = re.compile(r"CL_FID08_(\d{4})\.geojson$")   # matches e.g. CL_FID08_1990.geojson -> captures 1990

year_files = {}
for f in CENTERLINE_DIR.glob("CL_FID08_*.geojson"):    # loop over every matching file in the folder
    m = pattern.search(f.name)
    if m:
        year_files[int(m.group(1))] = f                # store year -> file path

print(f"Found {len(year_files)} yearly centerline files.")
print("Years:", sorted(year_files.keys()))

Found 34 yearly centerline files.
Years: [1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]


---
## Step 4 — Function: count channel crossings per ring for one year

Merges all channel-line segments into one geometry (so a braided network with many separate line features is treated as a single continuous shape), then intersects each ring with it and counts the resulting points.

In [11]:
def count_channels_per_ring(centerline_gdf, circles_gdf):
    # Reproject the centerlines to match the circle file's CRS, if needed
    if centerline_gdf.crs != circles_gdf.crs:
        centerline_gdf = centerline_gdf.to_crs(circles_gdf.crs)

    river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry

    results = []
    for _, ring in circles_gdf.iterrows():               # go ring by ring (5 km, 10 km, ...)
        ring_geom = ring.geometry                          # this ring's boundary line
        d = ring["distance"]                                # the distance label for this ring (meters)

        inter = ring_geom.intersection(river_union)         # where the ring crosses the channel network

        if inter.is_empty:
            n = 0                                             # ring doesn't touch any channel -> 0 crossings
        elif inter.geom_type == "Point":
            n = 1                                             # exactly one crossing point
        elif inter.geom_type == "MultiPoint":
            n = len(inter.geoms)                              # multiple distinct crossing points
        elif inter.geom_type == "GeometryCollection":
            n = sum(1 for g in inter.geoms if g.geom_type == "Point")   # count only point parts
        else:
            n = None                                          # e.g. a short overlapping line segment -
                                                                # flag for manual review instead of guessing

        results.append({"distance": d, "AF": n})

    return pd.DataFrame(results).sort_values("distance").reset_index(drop=True)

---
## Step 5 — Run for every year and save results to Excel (one sheet per year)

In [13]:
all_results = {}   # year -> AF(d) table, kept in memory too so you can inspect/plot without re-running

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for year in sorted(year_files.keys()):                     # process years in chronological order
        cl = gpd.read_file(year_files[year])                     # load that year's centerline network
        af_table = count_channels_per_ring(cl, circles)          # intersect against all 41 rings
        all_results[year] = af_table

        af_table.to_excel(writer, sheet_name=str(year), index=False)   # one sheet per year, matches your old format

        n_zero = (af_table["AF"] == 0).sum()                     # rings with no channel at all that year
        n_flagged = af_table["AF"].isna().sum()                  # rings needing manual review (edge-case geometry)
        print(f"Year {year}: total crossings={af_table['AF'].sum()}, "
              f"zero-crossing rings={n_zero}, flagged rings={n_flagged}")

print(f"\nSaved AF(d) counts for all years to: {OUTPUT_XLSX}")

C:\Users\rbe\AppData\Local\Temp\ipykernel_14252\1860426911.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry


Year 1988: total crossings=98, zero-crossing rings=0, flagged rings=0
Year 1989: total crossings=102, zero-crossing rings=0, flagged rings=0


C:\Users\rbe\AppData\Local\Temp\ipykernel_14252\1860426911.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry
C:\Users\rbe\AppData\Local\Temp\ipykernel_14252\1860426911.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry


Year 1990: total crossings=119, zero-crossing rings=0, flagged rings=0


C:\Users\rbe\AppData\Local\Temp\ipykernel_14252\1860426911.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry


...

Year 2019: total crossings=104, zero-crossing rings=0, flagged rings=0


C:\Users\rbe\AppData\Local\Temp\ipykernel_14252\1860426911.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry


Year 2020: total crossings=113, zero-crossing rings=0, flagged rings=0


C:\Users\rbe\AppData\Local\Temp\ipykernel_14252\1860426911.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  river_union = centerline_gdf.geometry.unary_union   # merge every channel segment into one geometry


Year 2021: total crossings=100, zero-crossing rings=0, flagged rings=0

Saved AF(d) counts for all years to: G:\Cahnnel Complexity Ganges\AF_Counts_AllYears.xlsx


---
## Step 6 — Quick sanity check: inspect any flagged rings before moving on

If any ring returned `AF = None`, it means the ring line overlapped a channel segment (ran briefly parallel to it) instead of crossing it cleanly at a point. This is rare but worth checking manually in QGIS for the specific year/ring before trusting the automated count for that cell.

In [7]:
for year, af_table in all_results.items():
    flagged = af_table[af_table["AF"].isna()]
    if len(flagged) > 0:
        print(f"Year {year}: flagged distances needing manual check -> {flagged['distance'].tolist()}")